In [1]:
import pandas as pd
import numpy as np
import pickle
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('../data/processed/model_data.csv')

# Load saved TF-IDF and numeric features
with open('../src/tfidf.pkl','rb') as f:
    tfidf = pickle.load(f)
with open('../src/numeric_features.pkl','rb') as f:
    NUMERIC_FEATURES = pickle.load(f)

# Rebuild feature matrix
text_features = tfidf.transform(df['clean_text'])
numeric_matrix = sp.csr_matrix(df[NUMERIC_FEATURES].values)
X = sp.hstack([text_features, numeric_matrix])
y = df['fraudulent']

print("Data loaded!")
print("X shape:", X.shape)
print("Fraud cases:", y.sum())

Data loaded!
X shape: (17880, 5008)
Fraud cases: 866


In [2]:
# Train test split - stratified to maintain 95/5 ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])
print("Fraud in test:", y_test.sum())

# Logistic Regression
print("\nTraining Logistic Regression...")
lr = LogisticRegression(class_weight='balanced', max_iter=1000)
lr.fit(X_train, y_train)
print("Done!")

# Random Forest
print("\nTraining Random Forest...")
rf = RandomForestClassifier(n_estimators=200,
                             class_weight='balanced',
                             random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
print("Done!")

Train size: 14304
Test size: 3576
Fraud in test: 173

Training Logistic Regression...
Done!

Training Random Forest...
Done!


In [3]:
for name, model in [('Logistic Regression', lr), 
                     ('Random Forest', rf)]:
    preds = model.predict(X_test)
    proba = model.predict_proba(X_test)[:,1]
    
    print(f"\n{'='*45}")
    print(f"  {name}")
    print(f"{'='*45}")
    print(classification_report(y_test, preds,
          target_names=['Real','Fake']))
    print(f"ROC-AUC: {roc_auc_score(y_test, proba):.3f}")


  Logistic Regression
              precision    recall  f1-score   support

        Real       0.99      0.95      0.97      3403
        Fake       0.46      0.88      0.60       173

    accuracy                           0.94      3576
   macro avg       0.72      0.92      0.79      3576
weighted avg       0.97      0.94      0.95      3576

ROC-AUC: 0.983

  Random Forest
              precision    recall  f1-score   support

        Real       0.98      1.00      0.99      3403
        Fake       0.99      0.56      0.72       173

    accuracy                           0.98      3576
   macro avg       0.98      0.78      0.85      3576
weighted avg       0.98      0.98      0.98      3576

ROC-AUC: 0.989


In [4]:
import pickle

with open('../src/model_lr.pkl', 'wb') as f:
    pickle.dump(lr, f)

with open('../src/model_rf.pkl', 'wb') as f:
    pickle.dump(rf, f)

print("Both models saved!")
print("  src/model_lr.pkl  → Logistic Regression (high recall)")
print("  src/model_rf.pkl  → Random Forest (high precision)")

Both models saved!
  src/model_lr.pkl  → Logistic Regression (high recall)
  src/model_rf.pkl  → Random Forest (high precision)
